# Physics-informed neural network (PINN): damped oscillator

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/pinn_oscillator)

A small neural network $\hat x(t; w)$ learns the motion of a damped mass on a spring from only ten noisy measurements in the first 40% of the interval. Adding the equation of motion $m\ddot x + \mu\dot x + kx = 0$ to the loss, evaluated with automatic differentiation at collocation points, lets it extrapolate where a plain network fails, and in inverse mode it also infers the unknown damping $\mu$.

**Vector form**

$$m\,\ddot x + \mu\,\dot x + k\,x = 0,\qquad \mathcal L(w, \mu) = \frac{1}{D}\sum_d \big(\hat x(t_d) - x_d\big)^2 + \lambda\,\frac{1}{C}\sum_c \big(m\,\ddot{\hat x}(t_c) + \mu\,\dot{\hat x}(t_c) + k\,\hat x(t_c)\big)^2 + \mathcal L_{0}$$

**Index (tensor) form**

$$h^{(l+1)}_{ni} = \tanh\!\big(W^{(l)}_{ij}\,h^{(l)}_{nj} + b^{(l)}_i\big) \qquad \hat x_n = W^{(L)}_{1j}\,h^{(L)}_{nj} + b^{(L)}_1 \qquad r_c = m\,\frac{\partial^2 \hat x_c}{\partial t_c^2} + \mu\,\frac{\partial \hat x_c}{\partial t_c} + k\,\hat x_c$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $t_n$ | `t_all = cat([t_c, t_d, 0])` | (N,) | one batch of input times: collocation points, measurement times and $t = 0$ |
| $W^{(l)}_{ij},\ b^{(l)}_i$ | `weights[l] = (W, b)` | (I, J), (I,) | trainable weights $w$ of layer $l$: sizes $1 \to 32 \to 32 \to 32 \to 1$ |
| $W^{(l)}_{ij}\,h^{(l)}_{nj} + b^{(l)}_i$ | `einsum('ij,nj->ni', W, h) + b` | (N, I) | repeated $j$ ⇒ summed over the neurons of the previous layer |
| $h^{(l+1)}_{ni}$ | `torch.tanh(layer(h, W, b))` | (N, I) | hidden activations; a smooth activation so that $\hat x$ has second derivatives |
| $\partial \hat x_n / \partial t_n$ | `autograd.grad(x, t, ones, create_graph=True)` | (N,) | gives $\dot{\hat x}$; called again on it gives $\ddot{\hat x}$ (exact derivatives of the network, no finite differences) |
| $r_c$ | `m * x_tt + mu * x_t + k * x` | (C,) | ODE residual at the collocation points: zero where $\hat x$ obeys the physics |
| $\mathcal L_{\mathrm{data}},\ \mathcal L_{\mathrm{phys}},\ \mathcal L_0$ | `loss_terms(weights, mu, t_d, x_d, t_c, m, k)` | () | data misfit, mean $r_c^2$, and the initial condition $(\hat x(0) - 1)^2 + (\dot{\hat x}(0)/\omega_0)^2$ |
| $\mu$ | `torch.tensor(mu_start, requires_grad=True)` | () | inverse mode: the damping is one more trainable parameter, updated by Adam with the weights |

A plain neural network is a flexible curve fitter: trained only on the measurements it can pass through them, but beyond the last data point nothing tells it what to do, so its prediction (dashed) drifts off. A physics-informed neural network (PINN) adds a second loss term, the mean squared residual $r_c = m\hat x'' + \mu\hat x' + k\hat x$ at collocation points $t_c$ that cover the whole interval, including where there is no data. The derivatives are exact derivatives of the network with respect to its input, computed by torch.autograd.grad with create_graph=True so that the loss can be backpropagated through them into the weights. Driving $r_c \to 0$ forces $\hat x$ to be a solution of the equation of motion everywhere; the data and the initial condition $x(0) = 1$, $\dot x(0) = 0$ then pick out the one solution that matches, so the network extrapolates correctly. The physics acts as a regulariser that replaces missing data. In the inverse problem the damping $\mu$ is unknown: it becomes a trainable parameter and the optimiser finds the value for which the data and the physics agree (watch it overshoot and settle). The exact answer is $x = e^{-\delta t}(\cos\omega t + \tfrac{\delta}{\omega}\sin\omega t)$ with $\delta = \mu/2m$ and $\omega = \sqrt{k/m - \delta^2}$. Practical points: the residual is of order $k$ times the error, so the weight $\lambda \approx 10^{-4}$ balances it against the data term (too large and the network settles on the trivial solution $\hat x = 0$, which also has zero residual); higher frequencies (larger $k$) need more iterations, because a network learns low frequencies first. Here the forward problem is solved only to illustrate the method: an ODE solver would be far faster. PINNs pay off for inverse problems, sparse and noisy data, and PDEs on awkward domains. Simplifications: a fixed mass $m = 1$, a fixed seed, and Adam with a cosine-decaying learning rate (no L-BFGS refinement).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/pinn_oscillator/main.py
"""Main loop: parameters → tensors → train the PINN (and a plain network for comparison) → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors

M = 1.0  # mass (kg): only the ratios μ/m and k/m matter
T_MAX = 1.0  # time interval [0, T_MAX] in s
WIDTHS = [1, 32, 32, 32, 1]  # input t → 3 hidden tanh layers of 32 neurons → output x̂
SEED = 0


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    k, mu_true = params["k"], params["mu"]
    inverse = params["mode"] == "inverse"
    lam_phys = 10.0 ** params["log10_lambda"]

    t_d, x_d = tensors.observations(
        params["n_data"], params["data_fraction"] * T_MAX, params["noise"], M, mu_true, k, SEED, dev
    )  # (D,), (D,)
    t_c = tensors.collocation_points(params["n_colloc"], T_MAX, dev)  # (C,)
    t_plot = tensors.plot_grid(T_MAX, dev)  # (300,)
    x_exact = equation.exact_solution(t_plot, M, mu_true, k)

    # PINN: data + physics + initial condition. In inverse mode μ starts from a wrong guess and is learned.
    mu_start = params["mu_guess"] if inverse else mu_true
    pinn = solve.train(
        tensors.init_weights(WIDTHS, SEED, dev), mu_start, inverse, t_d, x_d, t_c, M, k,
        lam_phys, params["n_iter"], params["lr"], t_plot, label="PINN",
    )  # fmt: skip
    # The same network (same starting weights) without the physics term, for comparison.
    plain = None
    if params["compare"] == "on":
        plain = solve.train(
            tensors.init_weights(WIDTHS, SEED, dev), mu_true, False, t_d, x_d, t_c, M, k,
            0.0, params["n_iter"], params["lr"], t_plot, label="no physics",
        )  # fmt: skip

    extrap = (t_plot > t_d.max()).cpu()  # beyond the last measurement
    exact = x_exact.cpu()

    def rel_err(x: torch.Tensor, mask=slice(None)) -> float:
        return (torch.norm(x[mask] - exact[mask]) / torch.norm(exact[mask])).item()

    print(f"m = {M} kg, k = {k} N/m, true μ = {mu_true} kg/s   ·   {params['n_data']} data points on "
          f"[0, {t_d.max().item():.2f}] s, {params['n_colloc']} collocation points on [0, {T_MAX}] s")  # fmt: skip
    data, phys, init = pinn["losses"][-1].tolist()
    print(f"PINN final loss: data {data:.2e}   λ·physics {lam_phys * phys:.2e}   initial {init:.2e}")
    x_pinn = pinn["snapshots"][-1]
    print(f"PINN        rel. L2 error: whole interval {rel_err(x_pinn):6.2%}   "
          f"extrapolation {rel_err(x_pinn, extrap):6.2%}")  # fmt: skip
    if plain is not None:
        x_plain = plain["snapshots"][-1]
        print(f"no physics  rel. L2 error: whole interval {rel_err(x_plain):6.2%}   "
              f"extrapolation {rel_err(x_plain, extrap):6.2%}")  # fmt: skip
    if inverse:
        err = abs(pinn["mu_final"] - mu_true) / mu_true
        print(f"inferred μ = {pinn['mu_final']:.3f} kg/s (started at {mu_start}, true {mu_true}, error {err:.1%})")

    return plot.animate(
        t_plot, x_exact, t_d, x_d, t_c, pinn, plain, lam_phys, mu_true, inverse, out_dir / "pinn_oscillator.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/pinn_oscillator/equation.py
r"""Physics → tensor translation.

The physics is a damped mass on a spring,

    m ẍ + μ ẋ + k x = 0,      x(0) = 1,  ẋ(0) = 0,

and the unknown function x(t) is replaced by a small neural network x̂(t; w). For a batch of time points n
each layer l of the network is

    h^{(l+1)}_{ni} = tanh( W^{(l)}_{ij} h^{(l)}_{nj} + b^{(l)}_i )        j summed (the neurons of layer l)

with h^{(0)}_n = t_n and a final linear layer x̂_n = W^{(L)}_{1j} h^{(L)}_{nj} + b^{(L)}_1.

The network is trained so that it (1) passes through a few measured points and (2) satisfies the equation of
motion at collocation times t_c. The ODE residual

    r_c = m x̂''_c + μ x̂'_c + k x̂_c

needs x̂' and x̂'': torch.autograd differentiates the network itself with respect to its input t.
"""

import math

import torch


def layer(h: torch.Tensor, W: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Pre-activation a_{ni} = W_{ij} h_{nj} + b_i.   h: (N, J), W: (I, J), b: (I,) → (N, I);  j summed."""
    return torch.einsum("ij,nj->ni", W, h) + b


def network(t: torch.Tensor, weights: list[tuple[torch.Tensor, torch.Tensor]]) -> torch.Tensor:
    """x̂(t; w): tanh hidden layers, then one linear output layer.   t: (N,) → x̂: (N,)."""
    h = t[:, None]  # (N, 1): the only input feature is the time
    for W, b in weights[:-1]:
        h = torch.tanh(layer(h, W, b))  # h^{(l+1)}_{ni} = tanh(W_ij h_nj + b_i)
    W, b = weights[-1]
    return layer(h, W, b)[:, 0]  # (N, 1) → (N,)


def time_derivative(y: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    """dy_n/dt_n for every point n (each y_n depends only on its own t_n).   (N,) → (N,).

    create_graph=True keeps the derivative itself differentiable, so we can take it again (x̂'')
    and later backpropagate the loss through it into the weights.
    """
    (dy,) = torch.autograd.grad(y, t, grad_outputs=torch.ones_like(y), create_graph=True)
    return dy


def residual(x: torch.Tensor, x_t: torch.Tensor, x_tt: torch.Tensor, m: float, mu, k: float) -> torch.Tensor:
    """r_c = m x''_c + μ x'_c + k x_c: zero wherever x(t) obeys the equation of motion.   (C,)."""
    return m * x_tt + mu * x_t + k * x


def network_and_derivatives(t: torch.Tensor, weights, second: bool = True):
    """x̂, x̂' and x̂'' at the times t, all (N,): differentiate the network itself with respect to its input.
    second=False skips x̂'' (returned as None) when only the slope is needed."""
    t = t.clone().requires_grad_(True)  # we differentiate with respect to t, so it must track gradients
    x = network(t, weights)
    x_t = time_derivative(x, t)
    x_tt = time_derivative(x_t, t) if second else None
    return x, x_t, x_tt


def exact_solution(t: torch.Tensor, m: float, mu: float, k: float) -> torch.Tensor:
    """Underdamped solution with x(0) = 1, ẋ(0) = 0:  x = e^{-δt} (cos ωt + (δ/ω) sin ωt),
    δ = μ / 2m,  ω = √(k/m - δ²)."""
    delta = mu / (2 * m)
    omega = math.sqrt(k / m - delta**2)
    return torch.exp(-delta * t) * (torch.cos(omega * t) + delta / omega * torch.sin(omega * t))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/pinn_oscillator/tensors.py
"""Build the concrete tensors: network weights, the few noisy measurements and the collocation times."""

import math

import torch

from .equation import exact_solution


def init_weights(widths: list[int], seed: int, device: torch.device) -> list[tuple[torch.Tensor, torch.Tensor]]:
    """Xavier-initialised weights for layer sizes widths = [1, 32, 32, 32, 1].

    Layer l has W^{(l)} of shape (widths[l+1], widths[l]) and b^{(l)} of shape (widths[l+1],).
    Every tensor requires gradients: these are the parameters w that training adjusts.
    """
    gen = torch.Generator().manual_seed(seed)  # same seed ⇒ same starting network
    weights = []
    for n_in, n_out in zip(widths[:-1], widths[1:]):
        bound = math.sqrt(6.0 / (n_in + n_out))
        W = (torch.rand(n_out, n_in, generator=gen) * 2 - 1) * bound  # uniform in [-bound, bound]
        b = torch.zeros(n_out)
        weights.append((W.to(device).requires_grad_(True), b.to(device).requires_grad_(True)))
    return weights


def observations(
    n_data: int, t_data_max: float, noise: float, m: float, mu: float, k: float, seed: int, device: torch.device
) -> tuple[torch.Tensor, torch.Tensor]:
    """n_data noisy measurements (t_d, x_d), evenly spaced on [0, t_data_max]: shapes (D,), (D,)."""
    gen = torch.Generator().manual_seed(seed + 1)
    t_d = torch.linspace(0.0, t_data_max, n_data)
    x_d = exact_solution(t_d, m, mu, k) + noise * torch.randn(n_data, generator=gen)
    return t_d.to(device), x_d.to(device)


def collocation_points(n_colloc: int, t_max: float, device: torch.device) -> torch.Tensor:
    """t_c, shape (C,): where the equation of motion is enforced, spread over the WHOLE interval."""
    return torch.linspace(0.0, t_max, n_colloc, device=device)


def plot_grid(t_max: float, device: torch.device, n: int = 300) -> torch.Tensor:
    """Fine time grid (n,) for drawing the curves and measuring the error."""
    return torch.linspace(0.0, t_max, n, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/pinn_oscillator/solve.py
"""Train the network with Adam on  loss = data term + λ_phys · physics term + initial-condition term."""

import math

import torch

from problems._common.progress import steps

from .equation import network, network_and_derivatives, residual


def loss_terms(weights, mu, t_d, x_d, t_c, m: float, k: float, physics_on: bool = True) -> dict[str, torch.Tensor]:
    """The three pieces of the loss (each a scalar).

    physics:  mean_c r_c²                                obey m x̂'' + μ x̂' + k x̂ = 0 at the collocation times
    data:     mean_d (x̂(t_d) - x_d)²                    fit the few measurements
    initial:  (x̂(0) - 1)² + (x̂'(0) / ω₀)²               start at rest at x = 1  (ω₀ = √(k/m) makes ẋ dimensionless)

    All times go through the network in ONE batch [t_c, t_d, 0]; then we slice the pieces back out.
    physics_on = False drops the collocation points and x̂'' (a plain network: cheaper, physics term = 0).
    """
    if not physics_on:
        t_c = t_c[:0]  # no collocation points
    C, D = len(t_c), len(t_d)
    t_all = torch.cat([t_c, t_d, torch.zeros(1, device=t_c.device)])  # (C + D + 1,)
    x, x_t, x_tt = network_and_derivatives(t_all, weights, second=physics_on)
    physics = torch.zeros((), device=t_c.device)
    if physics_on:
        r = residual(x[:C], x_t[:C], x_tt[:C], m, mu, k)  # (C,)
        physics = torch.mean(r**2)
    data = torch.mean((x[C : C + D] - x_d) ** 2)
    initial = (x[-1] - 1.0) ** 2 + (x_t[-1] / math.sqrt(k / m)) ** 2
    return {"data": data, "physics": physics, "initial": initial}


def train(
    weights,
    mu_start: float,
    learn_mu: bool,
    t_d: torch.Tensor,
    x_d: torch.Tensor,
    t_c: torch.Tensor,
    m: float,
    k: float,
    lam_phys: float,
    n_iter: int,
    lr: float,
    t_plot: torch.Tensor,
    n_snapshots: int = 50,
    label: str = "training",
) -> dict:
    """Run Adam for n_iter steps. Returns the loss history (n_iter, 3), μ history (n_iter,),
    and snapshots of x̂ on t_plot taken at the iterations in 'snap_iters'.

    learn_mu = True is the inverse problem: μ is one more trainable number, found from data + physics.
    lam_phys = 0 switches the physics off: a plain network fitted to the data only.
    """
    mu = torch.tensor(mu_start, device=t_d.device, requires_grad=learn_mu)
    groups = [{"params": [p for Wb in weights for p in Wb], "lr": lr}]
    if learn_mu:
        groups.append({"params": [mu], "lr": 10 * lr})  # one scalar: let it move faster than the weights
    opt = torch.optim.Adam(groups)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_iter, eta_min=lr / 20)  # smaller steps at the end

    snap_iters = sorted({round(s * (n_iter - 1) / max(1, n_snapshots - 1)) for s in range(n_snapshots)})
    snap_set = set(snap_iters)
    losses = torch.zeros(n_iter, 3)
    mus = torch.zeros(n_iter)
    snapshots = []
    for it in steps(n_iter, label):
        terms = loss_terms(weights, mu, t_d, x_d, t_c, m, k, physics_on=lam_phys > 0)
        loss = terms["data"] + lam_phys * terms["physics"] + terms["initial"]
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()

        losses[it] = torch.stack([terms["data"], terms["physics"], terms["initial"]]).detach().cpu()
        mus[it] = mu.detach().cpu()
        if it in snap_set:
            with torch.no_grad():
                snapshots.append(network(t_plot, weights).cpu())
    return {
        "losses": losses,  # columns: data, physics (mean r²), initial
        "mu": mus,
        "snap_iters": snap_iters,
        "snapshots": torch.stack(snapshots),  # (S, n_plot)
        "mu_final": mu.item(),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/pinn_oscillator/plot.py
"""Animate training: the network's guess x̂(t) closing in on the true motion, the loss terms, and μ (or the error)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

BLUE, ORANGE, RED, GREY = "#2563eb", "#ea580c", "#dc2626", "#9ca3af"


def animate(
    t_plot: torch.Tensor,
    x_exact: torch.Tensor,
    t_d: torch.Tensor,
    x_d: torch.Tensor,
    t_c: torch.Tensor,
    pinn: dict,
    plain: dict | None,
    lam_phys: float,
    mu_true: float,
    inverse: bool,
    path: Path,
) -> Path:
    t, ex = t_plot.cpu().numpy(), x_exact.cpu().numpy()
    snaps = pinn["snapshots"].numpy()  # (S, n_plot)
    snaps_plain = plain["snapshots"].numpy() if plain is not None else None
    iters = np.array(pinn["snap_iters"])  # iteration of each snapshot
    n_iter = len(pinn["mu"])
    stride = max(1, n_iter // 400)  # ~400 points per history curve are plenty for drawing
    it_hist = np.arange(0, n_iter, stride)
    loss = pinn["losses"][::stride].numpy()  # (n_hist, 3): data, physics, initial
    t_end = t_d.max().item()

    fig = plt.figure(figsize=(8.2, 4.2))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.5, 1])
    ax = fig.add_subplot(grid[:, 0])
    ax_l = fig.add_subplot(grid[0, 1])
    ax_m = fig.add_subplot(grid[1, 1])

    # --- the motion: exact (grey), data (dots), collocation points (ticks), PINN (bold), no physics (dashed) ---
    ax.set_xlim(0, t[-1])
    ax.set_ylim(-1.3, 1.4)
    ax.set_xlabel("time t (s)")
    ax.set_ylabel("displacement x")
    ax.set_title("damped oscillator: network vs exact", fontsize=10)
    ax.axvspan(0, t_end, color="#dbeafe", alpha=0.6, lw=0)
    ax.text(t_end / 2, 1.3, "data", ha="center", va="top", fontsize=8, color="#1e40af")
    ax.text((t_end + t[-1]) / 2, 1.3, "extrapolation", ha="center", va="top", fontsize=8, color="#6b7280")
    ax.plot(t, ex, color=GREY, lw=3, alpha=0.7, label="exact")
    ax.plot(t_c.cpu().numpy(), np.full(len(t_c), -1.22), "|", color="#16a34a", ms=7, label="collocation $t_c$")
    ax.plot(t_d.cpu().numpy(), x_d.cpu().numpy(), "o", color=RED, ms=4.5, zorder=5, label="noisy data")
    (line_pinn,) = ax.plot([], [], color=BLUE, lw=2.2, label="PINN")
    line_plain = None
    if snaps_plain is not None:
        (line_plain,) = ax.plot([], [], "--", color=ORANGE, lw=1.6, label="no physics")
    ax.legend(loc="lower right", fontsize=7, ncol=2, bbox_to_anchor=(1.0, 0.04))
    counter = ax.text(0.03, 0.1, "", transform=ax.transAxes, fontsize=9)

    # --- loss terms vs iteration (log scale) ---
    ax_l.set_yscale("log")
    ax_l.set_xlim(0, n_iter)
    scale = np.array([1.0, lam_phys, 1.0])  # draw the physics term as it enters the loss, λ·mean(r²)
    shown = np.maximum(loss * scale, 1e-12)
    ax_l.set_ylim(shown.min() * 0.5, shown.max() * 2)
    ax_l.minorticks_off()
    ax_l.set_title("loss terms", fontsize=10)
    ax_l.tick_params(labelsize=8)
    curves = [
        ax_l.plot([], [], color=RED, lw=1.2, label="data")[0],
        ax_l.plot([], [], color="#16a34a", lw=1.2, label="λ·physics")[0],
        ax_l.plot([], [], color="#9333ea", lw=1.2, label="initial")[0],
    ]
    ax_l.legend(loc="upper right", fontsize=7)

    # --- inverse: inferred μ vs iteration;  forward: relative error vs iteration ---
    ax_m.set_xlim(0, n_iter)
    ax_m.set_xlabel("iteration", fontsize=9)
    ax_m.tick_params(labelsize=8)
    if inverse:
        mu = pinn["mu"][::stride].numpy()
        ax_m.set_title("inferred damping μ", fontsize=10)
        ax_m.axhline(mu_true, color=GREY, ls="--", lw=1)
        ax_m.text(n_iter * 0.97, mu_true, f"true {mu_true:g}", ha="right", va="bottom", fontsize=8, color="#6b7280")
        ax_m.set_ylim(0, max(mu.max(), mu_true) * 1.15)
        (mu_line,) = ax_m.plot([], [], color=BLUE, lw=1.6)
    else:
        err = np.linalg.norm(snaps - ex, axis=1) / np.linalg.norm(ex)
        err_plain = np.linalg.norm(snaps_plain - ex, axis=1) / np.linalg.norm(ex) if snaps_plain is not None else None
        ax_m.set_title("relative L2 error vs exact", fontsize=10)
        ax_m.set_yscale("log")
        ax_m.minorticks_off()
        lo = min(err.min(), err_plain.min() if err_plain is not None else 1.0)
        ax_m.set_ylim(lo * 0.5, max(err.max(), err_plain.max() if err_plain is not None else 0) * 2)
        (err_line,) = ax_m.plot([], [], color=BLUE, lw=1.6)
        err_plain_line = ax_m.plot([], [], "--", color=ORANGE, lw=1.4)[0] if err_plain is not None else None

    def update(s: int):
        line_pinn.set_data(t, snaps[s])
        if line_plain is not None:
            line_plain.set_data(t, snaps_plain[s])
        upto = it_hist <= iters[s]
        for j, curve in enumerate(curves):
            curve.set_data(it_hist[upto], shown[upto, j])
        if inverse:
            mu_line.set_data(it_hist[upto], mu[upto])
            counter.set_text(f"iteration {iters[s] + 1}   μ = {mu[upto][-1]:.2f}")
        else:
            err_line.set_data(iters[: s + 1], err[: s + 1])
            if err_plain_line is not None:
                err_plain_line.set_data(iters[: s + 1], err_plain[: s + 1])
            counter.set_text(f"iteration {iters[s] + 1}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(snaps), 50), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'mode': 'inverse',
    'k': 225.0,
    'mu': 3.0,
    'mu_guess': 1.0,
    'n_data': 10,
    'data_fraction': 0.4,
    'noise': 0.02,
    'n_colloc': 40,
    'log10_lambda': -4.0,
    'n_iter': 4000,
    'lr': 0.01,
    'compare': 'on'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.pinn_oscillator")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.pinn_oscillator.main")
problem_dir = Path("problems/ai/pinn_oscillator")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
